<a href="https://colab.research.google.com/github/MAI-AIIU/advanced-programming/blob/main/First_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Building an AI-Ready Dataset Using Web Scraping, External Datasets, NumPy, and Pandas.

In this assignment, students will collect data from the web using web scraping and combine it with another external
dataset. They will clean, transform, analyze, and prepare the data using Pandas and NumPy.

## Scraping `books.toscrape.com` website
Loading the required libraries, which I use `requests` to send request and pull data and using `BeautifulSoup` to scrape data from the HTML and CSS data I have recieved via request

#### Load libraries

In [1]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

#### Define base_url and headers

In [2]:
BASE_URL = "https://books.toscrape.com/"

# we use this header to avoid flagging our request as automated by the website
headers = {
    "User-Agent": "Mozilla/5.0"
}

#### Define mappers to map data from original form to another form
in below example we map ratings from text to numbers

In [3]:
rating_map = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

#### Defining global variables

In [4]:
books = []
next_page = BASE_URL
page = 5 # I put this variable to only load first 5 pages.

In [5]:
while next_page:
    print(f"Scraping {next_page}")

    response = requests.get(next_page, headers=headers)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    for book in soup.select("article.product_pod"):

        title = book.h3.a["title"]
        price = book.select_one("p.price_color").text.strip()

        availability = (
            book.select_one("p.instock.availability")
            .get_text(strip=True)
        )

        rating = rating_map[
            book.select_one("p.star-rating")["class"][1]
        ]

        # Product URL
        relative_url = book.h3.a["href"]
        product_url = urljoin(next_page, relative_url)

        # Visit product page to get category
        detail_response = requests.get(product_url, headers=headers)
        detail_response.raise_for_status()

        detail_soup = BeautifulSoup(detail_response.text, "html.parser")

        category = (
            detail_soup.select("ul.breadcrumb li a")[2]
            .get_text(strip=True)
        )

        books.append({
            "Title": title,
            "Price": price,
            "Rating": rating,
            "Availability": availability,
            "Category": category,
            "Product Link": product_url
        })

    # Find next page
    next_button = soup.select_one("li.next a")

    if next_button and page > 1:
        page -= 1
        next_page = urljoin(next_page, next_button["href"])
    else:
        next_page = None

print(f"Scraped {len(books)} books.")
print("Saved to books.csv.")

Scraping https://books.toscrape.com/
Scraping https://books.toscrape.com/catalogue/page-2.html
Scraping https://books.toscrape.com/catalogue/page-3.html
Scraping https://books.toscrape.com/catalogue/page-4.html
Scraping https://books.toscrape.com/catalogue/page-5.html
Scraped 100 books.
Saved to books.csv.


#### Export to CSV file

In [6]:
import csv

with open("toscrape_books.csv", "w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(
        file,
        fieldnames=[
            "Title",
            "Price",
            "Rating",
            "Availability",
            "Category",
            "Product Link"
        ]
    )
    writer.writeheader()
    writer.writerows(books)